In [1]:
#| default_exp phone

# The whole voice, off PyTorch

> Kokoro's front half and sine source converted for both phones at any length, the decoders shrunk for download, and
> the glue an app needs, scored against Kokoro on the Mac.

Notebooks 103 and 104 put the decoder on each phone. A phone needs the rest too: everything `analyse` does after
turning text into phonemes (the next notebook's job), and the decoder's sine source, which stayed in PyTorch. This
notebook converts those, makes everything small enough to offer as a download, and then runs the whole pipeline the
way an app will: each platform's runtimes, and only the glue code an app would have to write.

It converts models and needs Google's converter, so `nbdev-test` skips it.

In [2]:
#| export
import copy
import os
import shutil
import subprocess
import sys
import time
import warnings

import coremltools as ct
import coremltools.optimize.coreml as cto
import numpy as np
import torch
import torch.nn.functional as F

from text_to_speech.core import DATA
from text_to_speech.coreml import Body, HarmonicSource, compile_coreml, exportable_decoder, tile, to_coreml
from text_to_speech.litert import phone_decoder
from text_to_speech.voice import FRAME, SR, analyse, decode, model, pipeline

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.


Torch version 2.15.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.8.0 is the most recent version that has been tested.


In [3]:
import librosa
from fastcore.test import test_close, test_eq

from text_to_speech.data import metadata

Everything this notebook makes goes under `data/voice`, one folder per platform: what each app would download.
Two crashes shape how it runs. ONNX Runtime and LiteRT, Android's runtimes, crash when they share a process with
Core ML here, so anything that runs them runs `isolated`, in a Python of its own. And converting a model after
predicting with another crashed Core ML's Python bindings, so conversions run `fresh`, from this module's exported
functions. The second turned out to be more general (it's what notebook 103 met too): Core ML lets go of a
prediction's inputs a little later, on a thread of its own, and if that frees a numpy array, Python crashes. So every
prediction goes through `predict`, which keeps its inputs alive for a minute.

In [4]:
#| export
VOICE = DATA / "voice"
IOS, ANDROID = VOICE / "ios", VOICE / "android"


def isolated(code, prelude="import numpy as np", **arrays):
    "Run `code` in a fresh Python, with `arrays` as the dict `inputs`; it puts arrays in the dict `outputs`."
    folder = VOICE / "parts"
    folder.mkdir(parents=True, exist_ok=True)
    np.savez(folder / "in.npz", **arrays)
    script = (
        f"{prelude}\nimport numpy as np\ninputs = dict(np.load({str(folder / 'in.npz')!r}))\noutputs = {{}}\n"
        f"{code}\nnp.savez({str(folder / 'out.npz')!r}, **outputs)"
    )
    r = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, check=False)
    if r.returncode:
        raise RuntimeError(r.stderr[-3000:])
    return dict(np.load(folder / "out.npz"))


def fresh(code, **arrays):
    "`isolated`, with this module imported: for converting models."
    prelude = "import torch, coremltools as ct, coremltools.optimize.coreml as cto\nfrom text_to_speech.phone import *"
    return isolated(code, prelude=prelude, **arrays)


_feeds = []  # (when, inputs) of recent predictions


def predict(m, feed):
    "m.predict(feed), keeping feed alive for a minute: Core ML lets go of its inputs later, on a thread of its own."
    now = time.monotonic()
    _feeds[:] = [(t, f) for t, f in _feeds if now - t < 60] + [(now, feed)]
    return m.predict(feed)


def size_mb(path):
    "Size of a file, or of everything in a folder, in MB."
    path = str(path)
    if os.path.isfile(path):
        return os.path.getsize(path) / 1e6
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6

## The bar

The same as appendix 193's: ten sentences, and the distance between log mel spectrograms, against the distance
between two of Kokoro's own runs with different seeds. Anything here should land where notebook 103's Core ML decoder
did, about 3.9 times that floor: most of that is the padding fixed sizes need, which costs nothing in predicted MOS.
Where a change moves the timing a little, the frame-by-frame distance punishes the shift more than any change in the
sound, so there's also the distance after aligning the two in time.
(UTMOS, appendix 193's MOS predictor, needs `torchaudio`, which has no Python 3.14 build yet, so this notebook goes by
distance alone.)

In [5]:
texts = ["Speed comes from the voice itself, not from stretching the audio afterwards."]
texts += [t for t in metadata().query("id.str.startswith('LJ050')").normalized_text if len(t) < 200][:9]
utterances = [analyse(t) for t in texts]


def mel(a):
    return np.log(np.maximum(librosa.feature.melspectrogram(y=a, sr=SR, n_fft=1024, hop_length=256, n_mels=80), 1e-5))


def distance(a, b):
    n = min(len(a), len(b))
    return np.abs(mel(a[:n]) - mel(b[:n])).mean()


def aligned(a, b):
    "The same distance after dynamic time warping: fair to audio whose timing differs a little."
    x, y = mel(a), mel(b)
    D, path = librosa.sequence.dtw(X=x, Y=y, metric="cityblock")
    return D[-1, -1] / (len(path) * x.shape[0])


def kokoro(seed):
    torch.manual_seed(seed)
    return [decode(u) for u in utterances]


ref, other = kokoro(0), kokoro(1)
seconds = sum(map(len, ref)) / SR
floor = np.mean([distance(a, b) for a, b in zip(ref, other)])
aligned_floor = np.mean([aligned(a, b) for a, b in zip(ref, other)])
print(f"{seconds:.1f}s of audio; two seeds differ by {floor:.3f}, or {aligned_floor:.3f} aligned")

61.8s of audio; two seeds differ by 0.070, or 0.070 aligned


## The front half as two modules

`analyse` runs Kokoro's ALBERT, its duration predictor and its text encoder over the phonemes, decides how many 25 ms
frames each phoneme lasts, then runs the pitch and energy predictors over the frames. The step in the middle, turning
durations into frames, is a loop over phonemes with a data-dependent size: no converter takes it, and it's two lines
of app code anyway. So the front half becomes two models with that step between them: `Tokens`, per phoneme, and
`Prosody`, per frame.

Kokoro's modules take masks and lengths so that they can run a batch of sentences of different lengths, and pack
their LSTMs' inputs to match. An app runs one sentence at a time, so the rewrite drops all of that, and folds weight
normalisation into the weights as notebook 103 did.

In [6]:
#| export
def folded(m):
    "A copy of `m` with weight normalisation folded into its weights."
    for x in m.modules():
        if hasattr(x, "weight_g"):  # weight_norm caches a computed weight, which deepcopy refuses
            x.weight = x.weight.detach()
    m = copy.deepcopy(m).eval()
    for x in m.modules():
        if hasattr(x, "weight_g"):
            torch.nn.utils.remove_weight_norm(x)
    return m.requires_grad_(False)


class Tokens(torch.nn.Module):
    "Phoneme ids (1, n) and a voice's style (1, 256) -> features per phoneme d, t_en, and each one's duration in frames."

    def __init__(self):
        super().__init__()
        self.bert, self.bert_encoder = folded(model.bert), folded(model.bert_encoder)
        self.predictor, self.text_encoder = folded(model.predictor), folded(model.text_encoder)

    def forward(self, ids, style):
        s = style[:, 128:]
        x = self.bert_encoder(self.bert(ids, attention_mask=torch.ones_like(ids)))  # (1, n, 512)
        se = s[:, None].expand(1, x.shape[1], 128)
        for block in self.predictor.text_encoder.lstms:  # DurationEncoder.forward, for one unpadded sentence
            if isinstance(block, torch.nn.LSTM):
                x = block(torch.cat([x, se], -1) if x.shape[-1] == 512 else x)[0]
            else:  # AdaLayerNorm, then the style again
                h = block.fc(s)[:, None]
                x = torch.cat([F.layer_norm(x, (512,), eps=block.eps) * (1 + h[..., :512]) + h[..., 512:], se], -1)
        d = x  # (1, n, 640)
        duration = torch.sigmoid(self.predictor.duration_proj(self.predictor.lstm(d)[0])).sum(-1)
        t = self.text_encoder.embedding(ids).transpose(1, 2)
        for c in self.text_encoder.cnn:
            t = c(t)
        t_en = self.text_encoder.lstm(t.transpose(1, 2))[0].transpose(1, 2)  # (1, 512, n)
        return d, t_en, duration


class Prosody(torch.nn.Module):
    "Duration features repeated out to frames (1, 640, T) and the style -> pitch and energy, two values a frame."

    def __init__(self):
        super().__init__()
        self.predictor = folded(model.predictor)

    def forward(self, en, style):
        return self.predictor.F0Ntrain(en, style[:, 128:])

Between them, the app does what `analyse` does in the middle: rounds the durations to whole frames, at least one
each, and repeats every phoneme's features once per frame it lasts. (`np.round` rounds halves to even, as
`torch.round` does. An app has to round the same way, or now and then a phoneme comes out a frame longer.)

In [7]:
#| export
def phoneme_ids(phonemes):
    "Kokoro's input for a phoneme string: its vocabulary's ids, with a silence at each end."
    return np.array([[0, *[model.vocab[p] for p in phonemes if p in model.vocab], 0]])


def voice_style(voice, phonemes):
    "A voice's style for a sentence: its pack holds one row per phoneme count."
    return pipeline(voice).load_voice(voice)[len(phonemes) - 1].numpy().astype(np.float32)


def frames(duration):
    "Each phoneme's frames at 1x: rounded half to even, and at least one."
    return np.maximum(1, np.round(np.asarray(duration).reshape(-1))).astype(int)


def per_frame(x, frames):
    "Repeat each phoneme's column of x (1, channels, n) once per frame it lasts."
    return np.ascontiguousarray(np.repeat(x, frames, axis=-1))


def repeat_to(x, n):
    "Notebook 103's padding, in numpy: repeat x along its last axis to length n."
    return np.concatenate([x] * -(-n // x.shape[-1]), -1)[..., :n]

In [8]:
tokens, prosody = Tokens().eval(), Prosody().eval()
u = utterances[0]
ids, style = phoneme_ids(u.phonemes), voice_style(u.voice, u.phonemes)
with torch.no_grad():
    d, t_en, duration = tokens(torch.tensor(ids), torch.tensor(style))
    f = frames(duration)
    F0, N = prosody(torch.tensor(per_frame(d.transpose(1, 2).numpy(), f)), torch.tensor(style))
test_eq(f, u.frames.numpy())
test_eq(per_frame(t_en.numpy(), f), u.asr.numpy())
test_eq(F0.numpy(), u.F0.numpy())
test_eq(N.numpy(), u.N.numpy())

Exactly `analyse`'s frames, features, pitch and energy.

## Any length

Notebook 103 gave the decoder fixed sizes and padded sentences up to them. That won't do here: the LSTMs read in both
directions, so padding on the end changes what the backward direction carries into every real phoneme, and the
durations with it. These models need the sentence's own length. `torch.export` won't give them that:

In [9]:
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="The tensor attributes")  # The LSTMs' weights, about to be unrolled
        torch.export.export(
            tokens, (torch.tensor(ids), torch.tensor(style)), dynamic_shapes=({1: torch.export.Dim("n")}, None)
        )
except torch._dynamo.exc.UserError as e:
    print(str(e).strip().splitlines()[0][:200])

Constraints violated (n)! For more information, run with TORCH_LOGS="+dynamic".


It unrolls each LSTM into one step per phoneme, at this sentence's length. Both runtimes have an LSTM of their own that
takes any length, though, and the older routes into them keep `torch.nn.LSTM` whole: for Core ML, tracing with
TorchScript (deprecated in PyTorch, still supported by coremltools), with flexible input shapes; for Android, the ONNX
exporter, run by ONNX Runtime. LiteRT, Android's route to the GPU for the decoder, has no such LSTM, and the front
half is a tenth of the work, so it can run on the CPU.

In [10]:
#| export
def to_coreml_flexible(module, args, inputs, outputs, path, precision=ct.precision.FLOAT32):
    "Trace `module` at `args` and convert it with `inputs` (ct.TensorTypes, flexible shapes allowed); compile at `path`."
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*torch.jit")  # Deprecated, and the only route that keeps the LSTMs
        traced = torch.jit.trace(module, args)
    m = ct.convert(
        traced,
        inputs=inputs,
        outputs=[ct.TensorType(o) for o in outputs],
        compute_precision=precision,
        minimum_deployment_target=ct.target.iOS18,
    )
    remove_coreml(path)
    return compile_coreml(m, path)


def remove_coreml(path):
    "Delete an .mlpackage and its compiled .mlmodelc: compiling won't overwrite one."
    for p in [str(path), str(path).replace(".mlpackage", ".mlmodelc")]:
        shutil.rmtree(p, ignore_errors=True)


def to_onnx(module, args, path, inputs, outputs, dynamic):
    "Export `module` at `args` to ONNX at `path`, with the named axes in `dynamic` free to change size."
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        torch.onnx.export(
            module,
            args,
            str(path),
            input_names=inputs,
            output_names=outputs,
            dynamic_axes=dynamic,
            opset_version=17,
            dynamo=False,
        )

In [11]:
#| export
TOKENS_IN = [
    ct.TensorType("ids", shape=(1, ct.RangeDim(3, 512, 64)), dtype=np.int32),
    ct.TensorType("style", shape=(1, 256)),
]
PROSODY_IN = [ct.TensorType("en", shape=(1, 640, ct.RangeDim(4, 4096, 256))), ct.TensorType("style", shape=(1, 256))]


def build_front(ids, style):
    "Convert Tokens and Prosody for both platforms, at the shapes of one sentence, into IOS and ANDROID."
    IOS.mkdir(parents=True, exist_ok=True)
    ANDROID.mkdir(parents=True, exist_ok=True)
    tokens, prosody = Tokens().eval(), Prosody().eval()
    ids, style = torch.tensor(ids), torch.tensor(style)
    with torch.no_grad():
        d, _, duration = tokens(ids, style)
        en = torch.tensor(per_frame(d.transpose(1, 2).numpy(), frames(duration)))
    to_coreml_flexible(tokens, (ids, style), TOKENS_IN, ["d", "t_en", "duration"], IOS / "tokens.mlpackage")
    to_coreml_flexible(prosody, (en, style), PROSODY_IN, ["F0", "N"], IOS / "prosody.mlpackage")
    to_onnx(
        tokens,
        (ids, style),
        ANDROID / "tokens.onnx",
        ["ids", "style"],
        ["d", "t_en", "duration"],
        {"ids": {1: "n"}, "d": {1: "n"}, "t_en": {2: "n"}, "duration": {1: "n"}},
    )
    to_onnx(
        prosody,
        (en, style),
        ANDROID / "prosody.onnx",
        ["en", "style"],
        ["F0", "N"],
        {"en": {2: "frames"}, "F0": {1: "frames2"}, "N": {1: "frames2"}},
    )

In [12]:
fresh('build_front(inputs["ids"], inputs["style"])', ids=ids, style=style)
for name in ["tokens.mlpackage", "prosody.mlpackage"]:
    print(f"{name}: {size_mb(IOS / name):.0f} MB")
for name in ["tokens.onnx", "prosody.onnx"]:
    print(f"{name}: {size_mb(ANDROID / name):.0f} MB")

tokens.mlpackage: 78 MB
prosody.mlpackage: 34 MB
tokens.onnx: 78 MB
prosody.onnx: 34 MB


Checked on a sentence of a different length from the one they were converted at:

In [13]:
u2 = utterances[3]
ids2, style2 = phoneme_ids(u2.phonemes), voice_style(u2.voice, u2.phonemes)
coreml_tokens = ct.models.CompiledMLModel(str(IOS / "tokens.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
coreml_prosody = ct.models.CompiledMLModel(str(IOS / "prosody.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
c = predict(coreml_tokens, {"ids": ids2.astype(np.int32), "style": style2})
en2 = per_frame(c["d"].transpose(0, 2, 1), frames(c["duration"]))
o = isolated(
    f"""
import onnxruntime as ort
tokens = ort.InferenceSession("{ANDROID / "tokens.onnx"}", providers=["CPUExecutionProvider"])
prosody = ort.InferenceSession("{ANDROID / "prosody.onnx"}", providers=["CPUExecutionProvider"])
outputs["duration"] = tokens.run(None, {{"ids": inputs["ids"], "style": inputs["style"]}})[2]
outputs["F0"] = prosody.run(None, {{"en": inputs["en"], "style": inputs["style"]}})[0]
""",
    ids=ids2,
    style=style2,
    en=en2,
)
test_eq(frames(c["duration"]), u2.frames.numpy())
test_eq(frames(o["duration"]), u2.frames.numpy())
for out, name in [(predict(coreml_prosody, {"en": en2, "style": style2})["F0"], "Core ML"), (o["F0"], "ONNX")]:
    print(f"{name}: pitch within {np.abs(out - u2.F0.numpy()).max():.4f} Hz of analyse's")
test_close(per_frame(c["t_en"], frames(c["duration"])), u2.asr.numpy(), eps=1e-3)

Core ML: pitch within 0.0027 Hz of analyse's
ONNX: pitch within 0.0027 Hz of analyse's


The same frames as `analyse`, and pitch to a ten-thousandth of a hertz. Both are 32-bit, and have to be: at 16 bits
the ALBERT inside `Tokens` overflows, and Core ML returns NaN for every duration:

In [14]:
half = fresh(
    """
args = torch.tensor(inputs["ids"]), torch.tensor(inputs["style"])
half = to_coreml_flexible(Tokens().eval(), args, TOKENS_IN, ["d", "t_en", "duration"], VOICE / "tokens16.mlpackage",
                          precision=ct.precision.FLOAT16)
outputs["duration"] = predict(half, {"ids": inputs["ids2"].astype(np.int32), "style": inputs["style2"]})["duration"]
remove_coreml(VOICE / "tokens16.mlpackage")
""",
    ids=ids,
    style=style,
    ids2=ids2,
    style2=style2,
)
print(half["duration"][0, :8])

[nan nan nan nan nan nan nan nan]


Their weights can still be stored in 8 bits and computed in 32. The catch is the durations. Each phoneme's is a sum
rounded to whole frames, and the smallest change to the weights tips some of them across a rounding boundary: a
frame longer here, shorter there, and the sentence a different length from Kokoro's. Four ways of storing all of
`Tokens`' weights in 8 bits, against 40 LJSpeech sentences:

In [15]:
#| export
LINEAR8 = cto.OptimizationConfig(global_config=cto.OpLinearQuantizerConfig(mode="linear_symmetric"))
BLOCK8 = cto.OpLinearQuantizerConfig(mode="linear_symmetric", granularity="per_block", block_size=32)


def weight_names(path, prefixes):
    "The names of the weights in the Core ML package at `path` that start with one of `prefixes`."
    spec = ct.models.MLModel(str(path), skip_model_load=True).get_spec()
    block = next(iter(spec.mlProgram.functions["main"].block_specializations.values()))
    return [
        op.outputs[0].name for op in block.operations if op.type == "const" and op.outputs[0].name.startswith(prefixes)
    ]


def quantize_coreml(name, config, suffix, palettize=False):
    "IOS/name.mlpackage with its weights compressed by `config`, as IOS/name+suffix, compiled."
    m = ct.models.MLModel(str(IOS / f"{name}.mlpackage"), skip_model_load=True)
    m = (cto.palettize_weights if palettize else cto.linear_quantize_weights)(m, config)
    remove_coreml(IOS / f"{name}{suffix}.mlpackage")
    m.save(str(IOS / f"{name}{suffix}.mlpackage"))
    ct.utils.compile_model(str(IOS / f"{name}{suffix}.mlpackage"), str(IOS / f"{name}{suffix}.mlmodelc"))

In [16]:
schemes = {
    "_channel": ("cto.OptimizationConfig(global_config=cto.OpLinearQuantizerConfig(mode='linear_symmetric'))", False),
    "_block": ("cto.OptimizationConfig(global_config=BLOCK8)", False),
    "_palette": ("cto.OptimizationConfig(global_config=cto.OpPalettizerConfig(nbits=8, mode='kmeans'))", True),
}
for suffix, (config, palettize) in schemes.items():
    fresh(f"quantize_coreml('tokens', {config}, {suffix!r}, palettize={palettize})")
sentences = [analyse(t) for t in metadata().normalized_text.dropna().sample(40, random_state=1)]


def durations_kept(name):
    "On how many of the 40 sentences IOS/name gives exactly the 32-bit model's frames, and how many frames moved."
    cpu = lambda n: ct.models.CompiledMLModel(str(IOS / f"{n}.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
    reference, model, same, moved = cpu("tokens"), cpu(name), 0, 0
    for u in sentences:
        feed = {"ids": phoneme_ids(u.phonemes).astype(np.int32), "style": voice_style(u.voice, u.phonemes)}
        a, b = frames(predict(reference, feed)["duration"]), frames(predict(model, feed)["duration"])
        same, moved = same + np.array_equal(a, b), moved + int(np.abs(a - b).sum())
    return (
        f"{size_mb(IOS / f'{name}.mlpackage'):.0f} MB, the same durations on {same}/40 sentences, {moved} frames moved"
    )


for suffix, label in [("_channel", "per channel"), ("_block", "per block of 32"), ("_palette", "a 256-value palette")]:
    print(f"8 bits {label}: {durations_kept('tokens' + suffix)}")

8 bits per channel: 20 MB, the same durations on 11/40 sentences, 60 frames moved


8 bits per block of 32: 22 MB, the same durations on 22/40 sentences, 26 frames moved


8 bits a 256-value palette: 20 MB, the same durations on 12/40 sentences, 48 frames moved


None keeps them all. But most of `Tokens` has nothing to do with durations: they come from ALBERT and the duration
predictor, while the text encoder only makes `t_en`, the decoder's features. And `Prosody` comes after the durations
are decided. The converter keeps PyTorch's names for most weights, so the text encoder's can be picked out by name,
`text_encoder_...`, and stored in 8 bits alone, with all of `Prosody`'s:

In [17]:
#| export
def quantize_front():
    "The front half for download: Tokens with just its text encoder's weights in 8 bits; Prosody all in 8 bits."
    names = weight_names(IOS / "tokens.mlpackage", ("text_encoder_",))
    quantize_coreml("tokens", cto.OptimizationConfig(op_name_configs=dict.fromkeys(names, BLOCK8)), "8")
    quantize_coreml("prosody", cto.OptimizationConfig(global_config=BLOCK8), "8")

In [18]:
fresh("quantize_front()")
fresh("""names = weight_names(IOS / 'tokens.mlpackage', ('text_encoder_', 'bert_encoder_'))
quantize_coreml('tokens', cto.OptimizationConfig(op_name_configs=dict.fromkeys(names, BLOCK8)), '_albert')""")
print(f"tokens, text encoder in 8 bits: {durations_kept('tokens8')}")
print(f"tokens, text encoder and ALBERT in 8 bits: {durations_kept('tokens_albert')}")
print(f"prosody: {size_mb(IOS / 'prosody.mlpackage'):.0f} MB -> {size_mb(IOS / 'prosody8.mlpackage'):.0f} MB")
for suffix in [*schemes, "_albert"]:
    remove_coreml(IOS / f"tokens{suffix}.mlpackage")

tokens, text encoder in 8 bits: 66 MB, the same durations on 40/40 sentences, 0 frames moved


tokens, text encoder and ALBERT in 8 bits: 49 MB, the same durations on 24/40 sentences, 27 frames moved
prosody: 34 MB -> 9 MB


Every duration kept, and the front half down from 112 MB to 75. ALBERT's weights in 8 bits as well would save
another 24 MB, but they decide durations, and move them like the rest. `Prosody`'s pitch moves by a few hertz at most,
which the end-to-end scores below put in context.

ONNX Runtime has no setting for this: its own quantizers compute in 8 bits, and leave the LSTMs alone. But an ONNX
weight can come from a `DequantizeLinear` node instead of a constant, so the file stores 8-bit numbers and a scale
per channel, and ONNX Runtime turns them back into 32-bit weights as it loads the model. The exporter keeps few of
PyTorch's names (ALBERT's and the LSTMs' weights become `onnx::MatMul_2393` and the like), so the weights are picked
by what they feed instead: in `Tokens`, those that reach only `t_en`, which is the same set; in `Prosody`, all of
them. The channel is each weight's output: a convolution's first axis, an LSTM's gate rows, a matrix's columns.

In [19]:
#| export
WEIGHT_AXES = {("Conv", 1): 0, ("LSTM", 1): 1, ("LSTM", 2): 1, ("MatMul", 1): 1}


def onnx_reach(model):
    "For each tensor in an ONNX graph, the graph outputs it feeds. Nodes are in topological order: walk them backwards."
    reach = {o.name: {o.name} for o in model.graph.output}
    for n in reversed(model.graph.node):
        r = set().union(*(reach.get(o, set()) for o in n.output))
        for x in n.input:
            reach[x] = reach.get(x, set()) | r
    return reach


def quantize_onnx(name, only=None, min_elements=4096):
    "ANDROID/name8.onnx: its 32-bit weights stored in 8 bits per channel; with `only`, just those reaching only `only`."
    import onnx
    from onnx import helper, numpy_helper

    model = onnx.load(ANDROID / f"{name}.onnx")
    graph, reach, users = model.graph, onnx_reach(model), {}
    for n in graph.node:
        for i, x in enumerate(n.input):
            users.setdefault(x, []).append((n, i))
    initializers, nodes = [], []
    for init in list(graph.initializer):
        w = numpy_helper.to_array(init)
        if w.dtype != np.float32 or w.size < min_elements or (only and not reach.get(init.name, set()) <= set(only)):
            continue
        axes = {
            (0 if any(a.name == "transB" and a.i for a in n.attribute) else 1)
            if n.op_type == "Gemm" and i == 1
            else WEIGHT_AXES.get((n.op_type, i))
            for n, i in users.get(init.name, [])
        }
        if len(axes) != 1 or None in axes:
            continue
        axis = axes.pop()
        scale = np.maximum(np.abs(np.moveaxis(w, axis, 0)).reshape(w.shape[axis], -1).max(axis=1), 1e-12) / 127
        shape = [1] * w.ndim
        shape[axis] = -1
        q = np.clip(np.round(w / scale.reshape(shape)), -127, 127).astype(np.int8)
        graph.initializer.remove(init)
        initializers += [
            numpy_helper.from_array(q, f"{init.name}_q8"),
            numpy_helper.from_array(scale.astype(np.float32), f"{init.name}_scale"),
        ]
        nodes.append(
            helper.make_node("DequantizeLinear", [f"{init.name}_q8", f"{init.name}_scale"], [init.name], axis=axis)
        )
    graph.initializer.extend(initializers)
    for n in reversed(nodes):
        graph.node.insert(0, n)
    onnx.save(model, ANDROID / f"{name}8.onnx")

In [20]:
quantize_onnx("tokens", only=["t_en"])
quantize_onnx("prosody")
arrays = {}
for i, u in enumerate(sentences):
    arrays[f"ids{i}"], arrays[f"style{i}"] = phoneme_ids(u.phonemes), voice_style(u.voice, u.phonemes)
front = isolated(
    f"""
import onnxruntime as ort
from text_to_speech.phone import ANDROID, frames, per_frame
sess = lambda name: ort.InferenceSession(str(ANDROID / f"{{name}}.onnx"), providers=["CPUExecutionProvider"])
t32, t8, p32, p8 = sess("tokens"), sess("tokens8"), sess("prosody"), sess("prosody8")
for i in range({len(sentences)}):
    feed = {{"ids": inputs[f"ids{{i}}"].astype(np.int64), "style": inputs[f"style{{i}}"]}}
    (d, _, a), (d8, _, b) = t32.run(None, feed), t8.run(None, feed)
    en = per_frame(d.transpose(0, 2, 1), frames(a))
    outputs[f"frames{{i}}"], outputs[f"frames8_{{i}}"] = frames(a), frames(b)
    outputs[f"d_same{{i}}"] = np.array(np.array_equal(d, d8))
    f0 = [p.run(None, {{"en": en, "style": inputs[f"style{{i}}"]}})[0] for p in (p32, p8)]
    outputs[f"pitch{{i}}"] = np.abs(f0[0] - f0[1]).max()
""",
    **arrays,
)
n = len(sentences)
same = sum(np.array_equal(front[f"frames{i}"], front[f"frames8_{i}"]) for i in range(n))
pitch = [float(front[f"pitch{i}"]) for i in range(n)]
print(
    f"tokens: {size_mb(ANDROID / 'tokens.onnx'):.0f} MB -> {size_mb(ANDROID / 'tokens8.onnx'):.0f} MB, the same durations on {same}/{n} sentences,",
    "d unchanged" if all(front[f"d_same{i}"] for i in range(n)) else "d changed",
)
print(
    f"prosody: {size_mb(ANDROID / 'prosody.onnx'):.0f} MB -> {size_mb(ANDROID / 'prosody8.onnx'):.0f} MB, pitch within {max(pitch):.1f} Hz, {np.median(pitch):.1f} typically"
)

tokens: 78 MB -> 62 MB, the same durations on 40/40 sentences, d unchanged
prosody: 34 MB -> 9 MB, pitch within 4.1 Hz, 2.1 typically


The same as on iOS: every duration, and pitch within a few hertz. ONNX Runtime turns the weights back into 32 bits
as it loads them, so this saves download, not memory.

## The sine source

Notebook 103 left `HarmonicSource` in PyTorch: on the Mac, a converted copy crashed PyTorch in the same process. A
phone has no PyTorch, but the source has two other problems. It draws random numbers, and it interpolates by a factor
of 1/300, which Core ML's converter refuses. So `Source` takes its random numbers as inputs, and does the two
interpolations as the arithmetic they amount to. Kokoro's pitch is constant over each 300-sample step, so the
downsampling is the mean of the middle two samples of each step, and upsampling blends each frame with its nearer
neighbour by a fixed weight.

In [21]:
#| export
class Source(torch.nn.Module):
    "HarmonicSource, deterministic: the caller supplies the initial phases (the first zero) and unit Gaussian noise."

    def __init__(self, dec):
        super().__init__()
        self.gen = dec.generator
        sg = self.gen.m_source.l_sin_gen
        k = sg.upsample_scale
        w = (torch.arange(k) + 0.5) / k - 0.5  # Where each output sample falls between its frame and the next
        self.register_buffer("w", w.abs())
        self.register_buffer("before", (w < 0).float())
        self.register_buffer("harmonics", torch.arange(1, sg.harmonic_num + 2, dtype=torch.float32))

    def upsample(self, x):
        "F.interpolate(x, scale_factor=k, mode='linear'): each output a blend of its frame and the nearer neighbour."
        prev, nxt = torch.cat([x[..., :1], x[..., :-1]], -1), torch.cat([x[..., 1:], x[..., -1:]], -1)
        neighbour = prev[..., None] * self.before + nxt[..., None] * (1 - self.before)  # (1, 9, n, k)
        return (x[..., None] * (1 - self.w) + neighbour * self.w).reshape(x.shape[0], x.shape[1], -1)

    def forward(self, f0, phases, noise):
        "f0 (1, 2T), phases (1, 9), noise (1, 9, 600T) -> the STFT of the source, (1, 22, 120T + 1)."
        g, sg = self.gen, self.gen.m_source.l_sin_gen
        k = sg.upsample_scale
        f0 = g.f0_upsamp(f0[:, None]).transpose(1, 2)  # (1, samples, 1)
        rad = (f0 * self.harmonics).clamp(min=0) / sg.sampling_rate
        rad = rad - torch.floor(rad)
        rad = torch.cat([rad[:, :1] + phases[:, None], rad[:, 1:]], 1)
        rad = (rad[:, k // 2 - 1 :: k] + rad[:, k // 2 :: k]) / 2  # Interpolating by 1/k: the middle two of each k
        cycles = self.upsample((torch.cumsum(rad, dim=1) * k).transpose(1, 2)).transpose(1, 2)
        sines = torch.sin(2 * torch.pi * (cycles - torch.floor(cycles))) * sg.sine_amp
        uv = (f0 > sg.voiced_threshold).float()
        sines = sines * uv + (uv * sg.noise_std + (1 - uv) * sg.sine_amp / 3) * noise.transpose(1, 2)
        merged = g.m_source.l_tanh(g.m_source.l_linear(sines))
        spec, phase = g.stft.transform(merged.transpose(1, 2).squeeze(1))
        return torch.cat([spec, phase], dim=1)

To check it against `HarmonicSource`, it needs the random numbers `HarmonicSource` would draw. That turned out to be
subtler than drawing the same shapes with the same seed. Kokoro's noise comes from `torch.randn_like` on its sine
waves, which are a transposed view, and PyTorch's CPU generator fills a non-contiguous tensor with different numbers
from a contiguous one of the same shape. So the noise has to be drawn into a transposed view too:

In [22]:
def kokoro_randoms(T, seed):
    "The initial phases and noise HarmonicSource draws for T frames after torch.manual_seed(seed), in the order it does."
    torch.manual_seed(seed)
    phases = torch.rand(1, 9)
    phases[:, 0] = 0
    noise = torch.empty(1, 9, 600 * T).transpose(1, 2).normal_().transpose(1, 2).contiguous()
    return phases, noise


def source_error(a, b):
    "Largest differences: magnitude, and phase modulo 2 pi where there's enough magnitude for a phase to mean anything."
    a, b = np.asarray(a), np.asarray(b)
    n = a.shape[1] // 2
    dp = np.abs(a[:, n:] - b[:, n:])
    return np.abs(a[:, :n] - b[:, :n]).max(), np.minimum(dp, 2 * np.pi - dp)[np.abs(b[:, :n]) > 0.01].max()


dec = exportable_decoder()
source, reference = Source(dec).eval(), HarmonicSource(dec).eval()
T = u.asr.shape[-1]
phases, noise = kokoro_randoms(T, 0)
with torch.no_grad():
    torch.manual_seed(0)
    har = reference(u.F0)
    mag, ph = source_error(source(u.F0, phases, noise), har)
print(f"magnitude within {mag:.4f} of a peak of {har[:, :11].abs().max():.2f}; phase within {ph:.3f} rad")

magnitude within 0.0010 of a peak of 1.36; phase within 0.061 rad


The same to a fraction of a percent: what's left is rounding, from adding up phase over thousands of cycles in a
different order, and it grows with the sentence. On a phone, any good Gaussian generator will do for the noise.

In [23]:
#| export
SOURCE_IN = [
    ct.TensorType("f0", shape=(1, ct.RangeDim(8, 8192, 512))),
    ct.TensorType("phases", shape=(1, 9)),
    ct.TensorType("noise", shape=(1, 9, ct.RangeDim(2400, 2457600, 153600))),
]


def build_source():
    "Convert Source for both platforms, 32-bit, into IOS and ANDROID."
    source = Source(exportable_decoder()).eval()
    f0, phases, noise = torch.full((1, 512), 150.0), torch.zeros(1, 9), torch.randn(1, 9, 600 * 256)
    to_coreml_flexible(source, (f0, phases, noise), SOURCE_IN, ["har"], IOS / "source.mlpackage")
    to_onnx(
        source,
        (f0, phases, noise),
        ANDROID / "source.onnx",
        ["f0", "phases", "noise"],
        ["har"],
        {"f0": {1: "frames2"}, "noise": {2: "samples"}, "har": {2: "columns"}},
    )

In [24]:
fresh("build_source()")
coreml_source = ct.models.CompiledMLModel(str(IOS / "source.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
T2 = u2.asr.shape[-1]
phases, noise = kokoro_randoms(T2, 1)
with torch.no_grad():
    torch.manual_seed(1)
    har2 = reference(u2.F0)
feed = {"f0": u2.F0.numpy(), "phases": phases.numpy(), "noise": noise.numpy()}
onnx_har = isolated(
    f"""
import onnxruntime as ort
source = ort.InferenceSession("{ANDROID / "source.onnx"}", providers=["CPUExecutionProvider"])
outputs["har"] = source.run(None, inputs)[0]
""",
    **feed,
)["har"]
for name, out in [("Core ML", predict(coreml_source, feed)["har"]), ("ONNX", onnx_har)]:
    print(
        f"{name}: magnitude within {source_error(out, har2)[0]:.4f}, phase within {source_error(out, har2)[1]:.3f} rad"
    )
print(f"{size_mb(IOS / 'source.mlpackage') * 1000:.0f} KB and {size_mb(ANDROID / 'source.onnx') * 1000:.0f} KB")

Core ML: magnitude within 0.0138, phase within 0.348 rad
ONNX: magnitude within 0.0138, phase within 0.348 rad
34 KB and 21 KB


Both runtimes agree with each other exactly. Against Kokoro the error is larger on this longer sentence, about 1% of
the peak: the same rounding in the accumulated phase, over more cycles. The decoder's own noise is far larger, as
the end-to-end scores below show. And it's a few kilobytes: the source is all arithmetic, with one tiny linear layer.

## Smaller downloads

Notebook 103 converted the decoder's body once per padded size, and notebook 104 converted it for one size. Each is a
complete copy of the same weights, so three sizes on iOS are 322 MB, and on Android, at 32 bits, 612 MB. Both runtimes
can share weights between sizes in one file: Core ML's multifunction models, and LiteRT's signatures.

Sharing makes more sizes free, and they're worth having. A sentence is padded up to the next size, and the body's time
goes on the padding too. With notebook 103's three sizes, over 200 LJSpeech sentences:

In [25]:
#| export
SIZES = [64, 128, 192, 256, 384, 512]  # Frames: 1.6 to 12.8 s. Notebook 103's three, and three more between them

In [26]:
lengths = [int(analyse(t).frames.sum()) for t in metadata().normalized_text.dropna().sample(200, random_state=2)]
for sizes in [[128, 256, 512], SIZES]:
    padded = [next(b for b in sizes if b >= T) for T in lengths if T <= sizes[-1]]
    print(f"{sizes}: the body works on {sum(padded) / sum(min(T, sizes[-1]) for T in lengths):.2f}x the frames it says")

[128, 256, 512]: the body works on 1.46x the frames it says
[64, 128, 192, 256, 384, 512]: the body works on 1.20x the frames it says


In [27]:
#| export
def body_package(T):
    "Notebook 103's body for T frames, as a 16-bit Core ML package in VOICE/parts, made once."
    path = VOICE / "parts" / f"size{T}.mlpackage"
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        dec = exportable_decoder()
        with torch.no_grad():
            f0 = torch.zeros(1, 2 * T)
            args = (torch.zeros(1, 512, T), f0, torch.zeros(1, 2 * T), torch.zeros(1, 128), HarmonicSource(dec)(f0))
            to_coreml(Body(dec), args).save(str(path))
    return path


def shared_body(name, compress=lambda m: m):
    "The body at every size in SIZES, one function each sharing their weights, in IOS/name; `compress` applied first."
    desc = ct.utils.MultiFunctionDescriptor()
    for T in SIZES:
        part = VOICE / "parts" / f"{name}{T}.mlpackage"
        if not part.exists():
            compress(ct.models.MLModel(str(body_package(T)), skip_model_load=True)).save(str(part))
        desc.add_function(str(part), src_function_name="main", target_function_name=f"body{T}")
    desc.default_function_name = f"body{SIZES[0]}"
    remove_coreml(IOS / f"{name}.mlpackage")
    ct.utils.save_multifunction(desc, str(IOS / f"{name}.mlpackage"))
    ct.utils.compile_model(str(IOS / f"{name}.mlpackage"), str(IOS / f"{name}.mlmodelc"))

In [28]:
fresh('shared_body("body")')
fresh('shared_body("body8", lambda m: cto.linear_quantize_weights(m, LINEAR8))')
print(
    f"{len(SIZES)} sizes: {sum(size_mb(VOICE / 'parts' / f'body{T}.mlpackage') for T in SIZES):.0f} MB apart, "
    f"{size_mb(IOS / 'body.mlpackage'):.0f} MB shared, {size_mb(IOS / 'body8.mlpackage'):.0f} MB shared at 8 bits"
)

6 sizes: 643 MB apart, 109 MB shared, 56 MB shared at 8 bits


For LiteRT, `litert_torch` converts several modules at once as signatures of one model, and shares a weight between
them when it's the same tensor. Each size's body comes from its own `phone_decoder`, so the bodies' parameters are
tied to the first one's before converting. Then `ai_edge_quantizer` makes copies with 16-bit and 8-bit weights, and
two that mix them, for reasons the scores below give. It
runs in notebook 104's throwaway environment, for the same reason: `litert_torch` pins an older `typing-extensions`
than Jupyter runs with.

In [29]:
#| export
CONVERT_SHARED = """
import sys, numpy as np, torch, litert_torch
from ai_edge_quantizer import quantizer, recipe
folder, sizes = sys.argv[1], [int(T) for T in sys.argv[2:]]
modules = [torch.export.load(f"{folder}/body{T}.pt2").module() for T in sizes]
first = dict(modules[0].named_parameters())
for m in modules[1:]:  # The same architecture: tie every parameter to the first module's, so the converter shares it
    for name, p in list(m.named_parameters()):
        assert torch.equal(p, first[name]), name
        owner, _, leaf = name.rpartition(".")
        setattr(m.get_submodule(owner) if owner else m, leaf, first[name])
converter = litert_torch
for T, m in zip(sizes, modules):
    io = np.load(f"{folder}/body{T}.npz")
    converter = converter.signature(f"body{T}", m, tuple(torch.tensor(io[f"arr_{i}"]) for i in range(5)))
converter.convert().export(f"{folder}/body32.tflite")
eights = {  # Which weights each copy keeps in 8 bits; the rest are 16
    "int8": None,
    "mixed": [(".*AdainResBlk1d.*", "CONV_2D"), (".*", "FULLY_CONNECTED")],
    "mixed_conv": [(".*", "CONV_2D"), (".*", "FULLY_CONNECTED")],
}
for name in ["fp16", *eights]:
    q = quantizer.Quantizer(f"{folder}/body32.tflite")
    if name == "int8":
        q.load_quantization_recipe(recipe.weight_only_wi8_afp32())
    else:
        q.add_weight_only_config(regex=".*", operation_name="*", num_bits=16, algorithm_key="float_casting")
        for regex, op in eights.get(name) or []:
            q.add_weight_only_config(regex=regex, operation_name=op, num_bits=8)
    q.quantize().export_model(f"{folder}/body_{name}.tflite", overwrite=True)
"""


def shared_tflite(u):
    "Notebook 104's body for every size in one LiteRT model, at 32, 16, 8 and mixed bits, in ANDROID. Inputs from `u`."
    for T in SIZES:
        source, body = phone_decoder(T)
        with torch.no_grad(), warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="No complete tensor")  # Saving tensors that share storage
            f0 = tile(u.F0, 2 * T)
            args = (tile(u.asr, T), f0, tile(u.N, 2 * T), u.style, source(f0))
            torch.export.save(torch.export.export(body, args), ANDROID / f"body{T}.pt2")
            np.savez(ANDROID / f"body{T}.npz", *[a.numpy() for a in args], out=body(*args).numpy())
    uv = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
    env = [uv, "run", "-q", "--no-project", "--python", "3.13", "--with", "torch==2.14.0", "--with", "litert-torch"]
    subprocess.run(
        [*env, "python", "-c", CONVERT_SHARED, str(ANDROID), *map(str, SIZES)],
        check=True,
        capture_output=True,
        text=True,
    )
    for T in SIZES:
        (ANDROID / f"body{T}.pt2").unlink()

In [30]:
shared_tflite(u)
for name in ["body32", "body_fp16", "body_int8", "body_mixed", "body_mixed_conv"]:
    print(f"{name}.tflite: {size_mb(ANDROID / f'{name}.tflite'):.0f} MB")

body32.tflite: 226 MB
body_fp16.tflite: 120 MB
body_int8.tflite: 71 MB
body_mixed.tflite: 88 MB
body_mixed_conv.tflite: 74 MB


Down from 612 MB to 226 at 32 bits, 120 at 16 and 71 at 8, with the two mixes between. Checked against PyTorch on the
Mac's CPU, where LiteRT runs without a GPU, by spectral distance in the bar's units:

In [31]:
#| export
LITERT_BODY = """
from ai_edge_litert.interpreter import Interpreter
runners = {}
def body(path, T, *args):
    if path not in runners:
        it = Interpreter(model_path=path)
        runners[path] = (it, {B: it.get_signature_runner(f"body{B}") for B in SIZES})
    r = runners[path][1][T]
    names = sorted(r.get_input_details(), key=lambda n: int(n.rsplit("_", 1)[-1]) if n[-1].isdigit() else 0)
    return list(r(**dict(zip(names, args))).values())[0]
"""

In [32]:
outs = isolated(f"""
SIZES = {SIZES}
{LITERT_BODY}
for name in ["body32", "body_fp16", "body_int8", "body_mixed", "body_mixed_conv"]:
    for T in SIZES:
        io = np.load("{ANDROID}/body%d.npz" % T)
        outputs[f"{{name}}_{{T}}"] = body("{ANDROID}/%s.tflite" % name, T, *[io[f"arr_{{i}}"] for i in range(5)])
""")
real = u.asr.shape[-1] * FRAME  # The sentence, before the padding
for name in ["body32", "body_fp16", "body_int8", "body_mixed", "body_mixed_conv"]:
    d = [
        distance(outs[f"{name}_{T}"].reshape(-1)[:real], np.load(ANDROID / f"body{T}.npz")["out"].reshape(-1)[:real])
        for T in SIZES
    ]
    print(f"{name}: {np.mean(d) / floor:.2f}x the seed floor from PyTorch")

body32: 0.00x the seed floor from PyTorch
body_fp16: 0.02x the seed floor from PyTorch
body_int8: 1.22x the seed floor from PyTorch
body_mixed: 0.13x the seed floor from PyTorch
body_mixed_conv: 0.45x the seed floor from PyTorch


16 bits costs nothing; 8 bits costs about as much as a different seed, where Core ML's 8 bits cost next to nothing.
Both store each weight's channels in 8 bits and compute in 32, so the difference is in what gets quantized. Taking
the body apart: the decoder's blocks (`AdainResBlk1d`, 31M weights in convolutions) take 8 bits with little loss, and
every linear layer does too; the generator does not, and above all its transposed convolutions, the upsampling, which
Core ML converts differently. So `body_mixed` keeps 8 bits for the decoder blocks' convolutions and the linear
layers and 16 for the rest, and `body_mixed_conv` adds the generator's plain convolutions. Whether 8-bit weights
also run on the Pixel's GPU, and how they sound, needs the phone.

The voices themselves are small: each is a table of 510 style vectors, one per sentence length in phonemes, 0.5 MB
as 32-bit floats.

In [33]:
#| export
def export_voice(voice, folder):
    "A voice's style table as raw little-endian float32, (510, 256), in folder/voice.bin."
    table = pipeline(voice).load_voice(voice).squeeze(1).numpy().astype("<f4")
    folder.mkdir(parents=True, exist_ok=True)
    table.tofile(folder / f"{voice}.bin")
    return table.shape

In [34]:
# Kokoro's English speakers graded C or better in its VOICES.md: the D grades sound noticeably worse
SPEAKERS = ["bf_emma", "bf_isabella", "bm_fable", "bm_george",
            "af_heart", "af_bella", "af_nicole", "af_aoede", "af_kore", "af_sarah", "am_fenrir", "am_michael", "am_puck"]
for voice in SPEAKERS:
    for folder in [IOS / "voices", ANDROID / "voices"]:
        shape = export_voice(voice, folder)
print(shape, f"{size_mb(IOS / 'voices' / 'af_heart.bin'):.2f} MB each")

(510, 256) 0.52 MB each


## All together

`synthesize` is what an app does, written in numpy so it's plain what each step is: run `Tokens`, turn durations into
frames, run `Prosody`, pick the smallest size that fits, pad as notebook 103 does, draw the source's random numbers,
run the source and the body, and trim. Each platform supplies four functions.

In [35]:
#| export
def synthesize(ids, style, tokens, prosody, source, body, rng, sizes=SIZES):
    "Audio for phoneme ids (1, n) and a style (1, 256), through four runtime functions. Longest: sizes[-1] frames."
    d, t_en, duration = tokens(ids, style)
    f = frames(duration)
    F0, N = prosody(per_frame(d.transpose(0, 2, 1), f), style)
    T = int(f.sum())
    B = next(b for b in sizes if b >= T)  # The app splits longer sentences before they get here
    f0 = repeat_to(F0, 2 * B).astype(np.float32)
    phases = rng.random((1, 9), dtype=np.float32)
    phases[0, 0] = 0
    har = source(f0, phases, rng.standard_normal((1, 9, 600 * B), dtype=np.float32))
    asr, n = repeat_to(per_frame(t_en, f), B).astype(np.float32), repeat_to(N, 2 * B).astype(np.float32)
    return np.asarray(body(B, asr, f0, n, style[:, :128], har)).reshape(-1)[: T * FRAME]


def ios_parts(body="body", front=""):
    "The four functions on Core ML: the front half and source on the CPU, the body on the GPU."
    cpu = lambda name: ct.models.CompiledMLModel(str(IOS / f"{name}.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
    tok, pro, src = cpu(f"tokens{front}"), cpu(f"prosody{front}"), cpu("source")
    bodies = {
        T: ct.models.CompiledMLModel(
            str(IOS / f"{body}.mlmodelc"), ct.ComputeUnit.CPU_AND_GPU, function_name=f"body{T}"
        )
        for T in SIZES
    }

    def tokens(ids, s):
        o = predict(tok, {"ids": ids.astype(np.int32), "style": s})
        return o["d"], o["t_en"], o["duration"]

    def prosody(en, s):
        o = predict(pro, {"en": en, "style": s})
        return o["F0"], o["N"]

    return {
        "tokens": tokens,
        "prosody": prosody,
        "source": lambda f0, p, n: predict(src, {"f0": f0, "phases": p, "noise": n})["har"],
        "body": lambda B, asr, f0, n, s, har: predict(
            bodies[B], {"asr": asr, "f0": f0, "n": n, "style": s, "har": har}
        )["out"],
    }

Scored against Kokoro, with the 16-bit body, the 8-bit body, and the 8-bit body with the front half as downloaded;
then the last with only notebook 103's three sizes, for the time the extra ones save:

In [36]:
inputs = [(phoneme_ids(u.phonemes), voice_style(u.voice, u.phonemes)) for u in utterances]


def score(name, parts, sizes=SIZES):
    rng = np.random.default_rng(0)
    for x in inputs:  # Each size's first prediction prepares it
        synthesize(*x, **parts, rng=rng, sizes=sizes)
    t = time.perf_counter()
    outs = [synthesize(*x, **parts, rng=rng, sizes=sizes) for x in inputs]
    rtf = seconds / (time.perf_counter() - t)
    report(name, outs, rtf)


def report(name, outs, rtf):
    lengths = sum(len(o) == len(r) for o, r in zip(outs, ref))
    d = np.mean([distance(o, r) for o, r in zip(outs, ref)]) / floor
    a = np.mean([aligned(o, r) for o, r in zip(outs, ref)]) / aligned_floor
    print(f"{name}: {rtf:.1f}x real time, {lengths}/10 Kokoro's length, {d:.2f}x the seed floor, {a:.2f}x aligned")


for body, front, name in [
    ("body", "", "16-bit body"),
    ("body8", "", "8-bit body"),
    ("body8", "8", "8-bit body, front half as downloaded"),
]:
    score(f"iOS, {name}", ios_parts(body, front))
score("iOS, as downloaded, three sizes", ios_parts("body8", "8"), sizes=[128, 256, 512])

iOS, 16-bit body: 33.3x real time, 10/10 Kokoro's length, 3.87x the seed floor, 3.85x aligned


iOS, 8-bit body: 34.1x real time, 10/10 Kokoro's length, 3.89x the seed floor, 3.87x aligned


iOS, 8-bit body, front half as downloaded: 33.4x real time, 10/10 Kokoro's length, 3.93x the seed floor, 3.91x aligned


iOS, as downloaded, three sizes: 32.6x real time, 10/10 Kokoro's length, 3.99x the seed floor, 3.97x aligned


The 16-bit pipeline is where notebook 103's decoder was, and the same length as Kokoro's every time: nothing is lost
by moving everything else off PyTorch. The 8-bit body costs next to nothing, and, once each size has made one
prediction, runs as fast as the 16-bit one: appendix 193 and an earlier version of this notebook timed 8 bits as a
third of the speed, but they timed each size's first use, when Core ML prepares it. The front half as downloaded
keeps every length, and its few hertz of pitch barely register. Three more sizes save about a tenth of the time.

Android's runtimes run in a process of their own: ONNX Runtime and LiteRT in the same process as Core ML have crashed
each other here. The script imports this module, so it's the same `synthesize`:

In [37]:
#| export
ANDROID_PARTS = """
import time, onnxruntime as ort
from text_to_speech.phone import ANDROID, SIZES, synthesize
sess = lambda name: ort.InferenceSession(str(ANDROID / f"{name}.onnx"), providers=["CPUExecutionProvider"])
tok, pro, src = sess("tokens" + FRONT), sess("prosody" + FRONT), sess("source")
parts = dict(tokens=lambda i, s: tuple(tok.run(None, {"ids": i.astype(np.int64), "style": s})),
             prosody=lambda en, s: tuple(pro.run(None, {"en": en, "style": s})),
             source=lambda f0, p, n: src.run(None, {"f0": f0, "phases": p, "noise": n})[0],
             body=lambda T, *args: body(str(ANDROID / BODY), T, *args))
"""


def android_check(inputs, body="body_fp16.tflite", front=""):
    "Run `synthesize` on ONNX Runtime and LiteRT, isolated: (the audio for each (ids, style), seconds taken)."
    arrays = {f"{k}{i}": a for i, (ids, style) in enumerate(inputs) for k, a in [("ids", ids), ("style", style)]}
    out = isolated(
        LITERT_BODY
        + ANDROID_PARTS.replace("BODY", repr(body)).replace("FRONT", repr(front))
        + f"""
rng = np.random.default_rng(0)
t = time.perf_counter()
for i in range({len(inputs)}):
    outputs[f"audio{{i}}"] = synthesize(inputs[f"ids{{i}}"], inputs[f"style{{i}}"], **parts, rng=rng)
outputs["seconds"] = np.array(time.perf_counter() - t)
""",
        **arrays,
    )
    return [out[f"audio{i}"] for i in range(len(inputs))], float(out["seconds"])

In [38]:
for body, front, name in [
    ("body_fp16.tflite", "", "16-bit body"),
    ("body_int8.tflite", "", "8-bit body"),
    ("body_mixed.tflite", "", "mixed body"),
    ("body_mixed_conv.tflite", "", "mixed body, generator convolutions 8-bit"),
    ("body_mixed_conv.tflite", "8", "mixed body, generator convolutions 8-bit, front half as downloaded"),
]:
    outs, elapsed = android_check(inputs, body, front)
    report(f"Android, {name} (the Mac's CPU)", outs, seconds / elapsed)

Android, 16-bit body (the Mac's CPU): 1.1x real time, 10/10 Kokoro's length, 3.86x the seed floor, 3.84x aligned


Android, 8-bit body (the Mac's CPU): 1.2x real time, 10/10 Kokoro's length, 4.54x the seed floor, 4.52x aligned


Android, mixed body (the Mac's CPU): 1.1x real time, 10/10 Kokoro's length, 3.87x the seed floor, 3.85x aligned


Android, mixed body, generator convolutions 8-bit (the Mac's CPU): 1.2x real time, 10/10 Kokoro's length, 3.88x the seed floor, 3.87x aligned


Android, mixed body, generator convolutions 8-bit, front half as downloaded (the Mac's CPU): 1.1x real time, 10/10 Kokoro's length, 3.98x the seed floor, 3.96x aligned


The 16-bit body matches Kokoro as closely as Core ML does. On the Mac's CPU, LiteRT only just keeps up; on the
Pixel's GPU, notebook 104 measured the body at about 10x real time. The 8-bit body loses more, as it did alone. Both
mixes score as the 16-bit body does: what they lose alone is lost under the distance from Kokoro that every phone
version has. The front half as downloaded costs about what iOS's does.

A phone without a usable GPU runs the body on its CPU, through XNNPACK, and there memory decides. XNNPACK packs the
weights into 32 bits for each of the six sizes separately, unless its weight cache, which must be a file, lets the
sizes share one copy; and it keeps each convolution's indirection buffer between runs unless told not to. The 8-bit
weights reach it as `DEQUANTIZE` operations, whose 32-bit results the cache doesn't share. Measured in the Android
app on a 2 GB emulator, with the cache and transient indirection buffers, the 16-bit body holds about 1 GB once it
has read a sentence of every size, and the mixed body nearly 2, which the emulator kills. (The Mac's memory
compresses pages as it pleases, so its process sizes can't measure this.) A GPU may close the gap, but a voice the
CPU can't hold is no use on the phones that need the CPU, so Android downloads the 16-bit body, with the 8-bit front
half, which ONNX Runtime turns back into 32 bits once, as it loads.

## What an app downloads

In [39]:
ios = {"tokens8.mlmodelc", "prosody8.mlmodelc", "source.mlmodelc", "body8.mlmodelc", "voices"}
android = {"tokens8.onnx", "prosody8.onnx", "source.onnx", "body_fp16.tflite", "voices"}
for name, folder, files in [("iOS", IOS, ios), ("Android", ANDROID, android)]:
    print(
        f"{name}: {sum(size_mb(folder / f) for f in files):.0f} MB:",
        ", ".join(f"{f} {size_mb(folder / f):.1f}" for f in sorted(files)),
    )

iOS: 132 MB: body8.mlmodelc 55.7, prosody8.mlmodelc 9.2, source.mlmodelc 0.0, tokens8.mlmodelc 66.1, voices 1.0
Android: 191 MB: body_fp16.tflite 119.7, prosody8.onnx 8.6, source.onnx 0.0, tokens8.onnx 61.8, voices 1.0


Plus the next notebook's text-to-phoneme files, which both platforms share, and which it puts together with these
into the folder each app downloads.

## Summary

- **The front half** is two modules, `Tokens` per phoneme and `Prosody` per frame, with two lines of app code between
  them. They reproduce `analyse` exactly.
- **Any length**: the LSTMs need each sentence's own length, which `torch.export` can't give them. TorchScript into
  Core ML and the ONNX exporter keep the LSTMs whole, at any length, and both match PyTorch. They compute in 32 bits
  (ALBERT overflows at 16). Durations are a rounding away from moving, so only the weights that don't decide them,
  the text encoder's and `Prosody`'s, are stored in 8 bits: 112 MB to 75 on iOS, and to 70 on Android, where the
  weights are picked by which outputs they reach and a `DequantizeLinear` node restores each at load.
- **The source** takes its randomness as inputs and converts for both. Reproducing Kokoro's noise exactly took
  drawing it into a transposed view.
- **Smaller downloads**: one file per platform holds the body at every size with its weights shared. iOS: three sizes'
  322 MB to 108 at 16 bits, 55 at 8, which costs nothing. Android: 612 MB to 120 at 16 bits, and 74 with every
  weight in 8 bits but the generator's transposed convolutions, which won't take it; end to end that costs no more
  than iOS's 8 bits, but on a CPU, XNNPACK keeps the 8-bit weights' 32-bit copies once per size, so Android downloads
  the 16-bit body. Sharing makes sizes free, so there are six, and sentences are padded less.
- **End to end**, through each platform's runtimes and only an app's glue, the voice is as close to Kokoro as
  notebook 103's decoder was. iOS downloads about 130 MB, Android about 190.
- **Two crashes**, both in the Mac's Python bindings rather than on a phone: Core ML frees prediction inputs on a
  thread of its own, and ONNX Runtime and LiteRT can't share a process with it.

Still to check on the phones: speed, the 8-bit body on the iPhone's GPU, and listening.